In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.callbacks import ModelCheckpoint

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
if devices := tf.config.list_physical_devices("GPU"):
    print(f"Running on {devices}")
else:
    print("Running on CPU")

In [ ]:
# load API key
try:
    from google.colab import userdata
    IN_COLAB = True
except:
    IN_COLAB = False


if IN_COLAB:
    %pip install -q zarr xarray fsspec aiohttp earthkit
    cdsapi_key = userdata.get("CDS-API-KEY")
else:
    import os

    %load_ext dotenv
    %dotenv

    cdsapi_key = os.getenv("CDS-API-KEY")


In [ ]:
import xarray as xr

# https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land

# Choose chunking approach
# "geo" for Geo-chunked stores for access optimised along the time dimension (e.g. time-series at a point)
# "time" for Time-chunked stores for access optimised in spatial dimensions (e.g. a global map)
chunks = "geo" # or "time"

# Soil temperature
soil_temperature_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-006/arco/reanalysis_era5_land/sfc-soil-temperature/{chunks}Chunked.zarr"
# 2m temperature
var_2m_temperature_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-007/arco/reanalysis_era5_land/sfc-2m-temperature/{chunks}Chunked.zarr"
# Soil water
soil_water_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-005/arco/reanalysis_era5_land/sfc-soil-water/{chunks}Chunked.zarr"
# Radiation and heat
radiation_and_heat_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-010/arco/reanalysis_era5_land/sfc-radiation-heat/{chunks}Chunked.zarr"
# Snow
snow_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-030/arco/reanalysis_era5_land/sfc-snow/{chunks}Chunked.zarr"
# Wind
wind_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-008/arco/reanalysis_era5_land/sfc-wind/{chunks}Chunked.zarr"
# Pressure and precipitation
pressure_and_precipitation_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-009/arco/reanalysis_era5_land/sfc-pressure-precipitation/{chunks}Chunked.zarr"
# Skin temperature
skin_temperature_url = f"https://arco.datastores.ecmwf.int/cadl-arco-{chunks}-043/arco/reanalysis_era5_land/sfc-skin-temperature/{chunks}Chunked.zarr"

datasets = [
    xr.open_zarr(
        var_2m_temperature_url,
        consolidated=True,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        }
    ),
    xr.open_zarr(
        wind_url,
        consolidated=True,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        }
    ),
    xr.open_zarr(
        pressure_and_precipitation_url,
        consolidated=True,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        }
    )
]

# only use data from Frankfurt am Main
ds = xr.merge(datasets)
# ds = ds_wind
ds = ds.sel(latitude=50.13, longitude=8.69, method="nearest")

features = {
    "d2m": "2m dewpoint in Kelvin",
    "t2m": "2m temperature in Kelvin",
    "u10": "10m wind eastward component i metre per second",
    "v10": "10m wind northward component i metre per second",
    "sp": "surface pressure in Pascal",
    "tp": "total precipitation in metre",
}

# Inspect the variables
ds.load()
ds

In [ ]:
# plot raw example data

from earthkit import plots as ekp
from earthkit import transforms as ekt

import warnings
warnings.filterwarnings(
    "ignore",
    message="TimeSeries is an experimental new feature*"
)

def plot_variable(variable_name, title):
    time = slice("2025-01-01", "2026-01-01")

    plot_data_hourly = ds[variable_name].sel(time=time)
    plot_data_monthly = ekt.temporal.monthly_mean(plot_data_hourly)

    chart = ekp.TimeSeries()

    chart.line(plot_data_hourly, label="Hourly")
    chart.line(plot_data_monthly, label="Monthly mean")

    chart.title(title)
    chart.legend()

    chart.show()

for name, title in features.items():
    plot_variable(name, title)

In [ ]:
# move to pandas Dataframe
df = ds.to_dataframe().drop(columns=["latitude", "longitude"])
df

In [ ]:
# split the data chonologically
n = len(df)

train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

In [ ]:
# define scalers for each variable
scalers = {}

for feature in features:
    scaler = StandardScaler()

    scaler.fit(
        train_df[[feature]]
    )

    scalers[feature] = scaler

# apply scalers
def scale_dataframe(data):
    scaled = np.zeros_like(data.values, dtype=np.float32)

    for i, feature in enumerate(features):
        scaled[:, i] = scalers[feature].transform(
            data[[feature]]
        ).flatten()

    return scaled


train_scaled = scale_dataframe(train_df)
val_scaled = scale_dataframe(val_df)
test_scaled  = scale_dataframe(test_df)

joblib.dump(scalers, "weather_scalers.pkl")

In [ ]:
# create encoder-decoder windows
INPUT_LEN = 168      # 7 days
OUTPUT_LEN = 24      # next 24 hours


def create_sequences(data, input_len, output_len):

    X = []
    y = []

    total = len(data)

    for i in range(total - input_len - output_len + 1):

        encoder = data[i : i + input_len]

        decoder = data[
            i + input_len :
            i + input_len + output_len
        ]

        X.append(encoder)
        y.append(decoder)

    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, INPUT_LEN, OUTPUT_LEN)
X_val, y_val = create_sequences(val_scaled, INPUT_LEN, OUTPUT_LEN)
X_test, y_test = create_sequences(test_scaled, INPUT_LEN, OUTPUT_LEN)

X_train.shape, y_train.shape

In [ ]:
# define model
n_features = X_train.shape[2]

inputs = Input(shape=(INPUT_LEN, n_features))

# Encoder
encoder = LSTM(
    128,
    activation="tanh"
)(inputs)

# Repeat context vector
decoder_input = RepeatVector(OUTPUT_LEN)(encoder)

# Decoder
decoder = LSTM(
    128,
    activation="tanh",
    return_sequences=True
)(decoder_input)

outputs = TimeDistributed(
    Dense(n_features)
)(decoder)

model = Model(inputs, outputs)

model.summary()

In [ ]:
# compile model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[RootMeanSquaredError(name="rmse"), "mae"]
)

In [ ]:
# train model
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop, checkpoint],
    shuffle=False
)

In [ ]:
history_dict = history.history

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss (MSE)
axes[0].plot(history_dict["loss"], label="Train")
if "val_loss" in history_dict:
    axes[0].plot(history_dict["val_loss"], label="Validation")
axes[0].set_title("Loss (MSE)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(True)

# RMSE
axes[1].plot(history_dict["rmse"], label="Train")
if "val_rmse" in history_dict:
    axes[1].plot(history_dict["val_rmse"], label="Validation")
axes[1].set_title("RMSE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("RMSE")
axes[1].legend()
axes[1].grid(True)

# MAE
axes[2].plot(history_dict["mae"], label="Train")
if "val_mae" in history_dict:
    axes[2].plot(history_dict["val_mae"], label="Validation")
axes[2].set_title("MAE")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("MAE")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# predict
y_pred = model.predict(X_test)

In [ ]:
# rescale features
def inverse_scale(data):
    inv = np.zeros_like(data)

    for i, feature in enumerate(features):
        inv[:, :, i] = scalers[feature].inverse_transform(
            data[:, :, i].reshape(-1,1)
        ).reshape(data.shape[0], data.shape[1])

    # undo precipitation transform
    tp_idx = features.index("tp")
    inv[:, :, tp_idx] = np.expm1(inv[:, :, tp_idx])

    return inv

y_pred_inv = inverse_scale(y_pred)
y_true_inv = inverse_scale(y_test)

In [ ]:
# compute metrics
for i, feature in enumerate(features):
    y_true = y_true_inv[:, :, i].ravel()
    y_hat = y_pred_inv[:, :, i].ravel()

    mae = mean_absolute_error(y_true, y_hat)
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))

    print(f"{feature:4s}  MAE={mae:.3f}   RMSE={rmse:.3f}")

In [ ]:
# plot predictions
sample = 0

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for i, ax in enumerate(axes.flat):
    ax.plot(
        y_true_inv[sample, :, i],
        label="True",
        linewidth=2
    )

    ax.plot(
        y_pred_inv[sample, :, i],
        "--",
        label="Prediction",
        linewidth=2
    )

    ax.set_title(features.keys()[i])
    ax.set_xlabel("Forecast step")
    ax.grid(True)

axes[0,0].legend()

plt.tight_layout()
plt.show()

In [ ]:
# overall metrics
overall_mae = mean_absolute_error(
    y_true_inv.ravel(),
    y_pred_inv.ravel()
)

overall_rmse = np.sqrt(
    mean_squared_error(
        y_true_inv.ravel(),
        y_pred_inv.ravel()
    )
)

print(f"Overall MAE : {overall_mae:.4f}")
print(f"Overall RMSE: {overall_rmse:.4f}")

In [ ]:
# evaluate by forecast horizon

for step in range(y_pred_inv.shape[1]):
    mae = mean_absolute_error(
        y_true_inv[:, step, :].ravel(),
        y_pred_inv[:, step, :].ravel()
    )
    print(f"Step {step+1:2d}: MAE = {mae:.3f}")